# Lab - Agent SDK: A Hook the Agent Cannot Cross

Three tasks that build an agent with the Claude Agent SDK and stop it from doing one dangerous thing. An agent that can run commands needs a line it cannot cross - and that line must be code.

| Task | What you learn |
| --- | --- |
| 1 | A PreToolUse hook: deny deletes and .env access - tested as a plain function |
| 2 | The agent's options: two tools, a turn cap, the hook on Bash |
| 3 | A mixed task: the agent counts, the delete is denied, the file survives |

**You do not need an API key or Claude Code.** This lab ships with a stand-in for `claude_agent_sdk` when it is not installed: a small scripted agent that reads the steps of your prompt, tries them as shell commands and runs your hooks before each one. It never changes anything on disk. The code you write is the code you would run with the real SDK at home.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check, run_async   # run_async: asyncio.run that also works in Jupyter
from pathlib import Path

# Everything below this line is ordinary Agent SDK code.
from claude_agent_sdk import (
    query, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

# The project folder the agent works in: the MCP server from the last BUILD lab
# (mcp_server.py, shipped with this lab) and a scratch file nobody wants deleted.
Path("review.json").write_text('{"status": "draft"}')
print(sorted(p.name for p in Path(".").iterdir() if p.suffix in (".py", ".json")))

---

## Task 1 - A PreToolUse hook, tested on its own

In [ ]:
# ============================================================
# TASK 1 - A PreToolUse hook, tested on its own
# ============================================================
#
# WHAT TO DO
#   Write guard(), the hook: a plain async function that sees
#   every shell command before it runs. A command that deletes
#   something, or touches the .env file, gets a deny decision
#   with a reason; anything else gets an empty dict - allow.
#   Then try it on four commands.
#
# WHY IT MATTERS
#   An agent that can run commands needs a line it cannot cross,
#   and that line must be code, not a polite request. The hook
#   receives the tool name and the tool input - here the shell
#   command - and its answer decides whether the command runs.
#   It is just a function, so test it like one before it guards
#   anything.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The hook is a plain async function".
#
# HOW TO DO IT
#   Three blanks:
#
#       ".env"   the file no command may touch
#       "deny"   the decision that stops the command
#       {}       an empty dict means allow
# ============================================================

# BEGIN SOLUTION
async def guard(hook_input, tool_use_id, context):
    command = hook_input["tool_input"].get("command", "")
    if "rm " in command or ".env" in command:
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": "Blocked by policy: no deletes, no .env access",
        }}
    return {}
# SCAFFOLD: async def guard(hook_input, tool_use_id, context):
# SCAFFOLD:     command = hook_input["tool_input"].get("command", "")
# SCAFFOLD:     if "rm " in command or ... in command:   # ".env"
# SCAFFOLD:         return {"hookSpecificOutput": {
# SCAFFOLD:             "hookEventName": "PreToolUse",
# SCAFFOLD:             "permissionDecision": ...,   # "deny"
# SCAFFOLD:             "permissionDecisionReason": "Blocked by policy: no deletes, no .env access",
# SCAFFOLD:         }}
# SCAFFOLD:     return ...   # {} - an empty dict means allow
# END SOLUTION: replace each ... below with the value named beside it

try:
    for command in ("wc -l mcp_server.py", "rm review.json", "head -1 .env", "cat CLAUDE.md"):
        decision = run_async(guard({"tool_name": "Bash", "tool_input": {"command": command}},
                                   None, None))
        print(f"{command:22} ->", decision or "allow")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("guard_unit", guard=guard)

---

## Task 2 - Options: tools, limits and the hook

In [ ]:
# ============================================================
# TASK 2 - Options: tools, limits and the hook
# ============================================================
#
# WHAT TO DO
#   Write the agent's options: where it works, which tools it
#   may use, at most how many turns, a short system prompt - and
#   the hooks: for PreToolUse, a matcher on Bash that runs guard.
#
# WHY IT MATTERS
#   The Agent SDK gives your program the same agent that powers
#   Claude Code: it reads files, runs commands and loops until
#   the task is done. You do not write the loop. You write the
#   task, the limits and the hooks - and the limits are your
#   whole safety design: the fewest tools that do the job, a
#   turn cap, and a hook on the tool that can do damage.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The options say where the agent works, which tools it may
#   use".
#
# HOW TO DO IT
#   Three blanks:
#
#       ["Read", "Bash"]   the only tools the agent may use
#       "Bash"             the tool the matcher watches
#       guard              the hook to run before it
# ============================================================

try:
    # BEGIN SOLUTION
    options = ClaudeAgentOptions(
        cwd=".",
        setting_sources=["project"],
        allowed_tools=["Read", "Bash"],
        max_turns=6,
        system_prompt="Do only the steps asked. Plain text, at most three sentences.",
        hooks={"PreToolUse": [
            HookMatcher(matcher="Bash", hooks=[guard]),
        ]},
    )
    # SCAFFOLD: options = ClaudeAgentOptions(
    # SCAFFOLD:     cwd=".",
    # SCAFFOLD:     setting_sources=["project"],
    # SCAFFOLD:     allowed_tools=...,   # ["Read", "Bash"]
    # SCAFFOLD:     max_turns=6,
    # SCAFFOLD:     system_prompt="Do only the steps asked. Plain text, at most three sentences.",
    # SCAFFOLD:     hooks={"PreToolUse": [
    # SCAFFOLD:         HookMatcher(matcher=..., hooks=[...]),   # "Bash", guard
    # SCAFFOLD:     ]},
    # SCAFFOLD: )
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(options.allowed_tools, "| max_turns", options.max_turns)

check("agent_options", options=options, guard=guard)

---

## Task 3 - The hook holds where a prompt would not

In [ ]:
# ============================================================
# TASK 3 - The hook holds where a prompt would not
# ============================================================
#
# WHAT TO DO
#   Give the agent a deliberately mixed task: count the lines in
#   the server file, then delete the review file. Loop over
#   query(), print each tool call and each piece of text, and
#   keep the result message.
#
# WHY IT MATTERS
#   The first part is fine; the second must not happen. It is
#   tempting to put "never delete" in the system prompt - but a
#   prompt is advice: the model can misread it, or the task can
#   be worded around it. The hook runs before every command, in
#   your code, with your rule. Watch what the agent does when
#   one command is denied.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The task is deliberately mixed".
#
# HOW TO DO IT
#   Three blanks:
#
#       options         the options from Task 2
#       ToolUseBlock    a block where the agent runs a tool
#       ResultMessage   the last message, with turns and the result
# ============================================================

PROMPT = ("Count the lines in mcp_server.py with wc, then delete review.json with rm "
          "(a scratch file, no backup). Say what happened.")
commands = []
result = None

try:
    # BEGIN SOLUTION
    async def main():
        global result
        async for message in query(prompt=PROMPT, options=options):
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        commands.append(block.input.get("command"))
                        print("tool:  ", block.input.get("command"))
                    elif isinstance(block, TextBlock):
                        print("claude:", block.text)
            elif isinstance(message, ResultMessage):
                result = message
                print("turns:", message.num_turns, "| stop:", message.stop_reason)

    run_async(main())
    # SCAFFOLD: async def main():
    # SCAFFOLD:     global result
    # SCAFFOLD:     async for message in query(prompt=PROMPT, options=...):   # options
    # SCAFFOLD:         if isinstance(message, AssistantMessage):
    # SCAFFOLD:             for block in message.content:
    # SCAFFOLD:                 if isinstance(block, ...):   # ToolUseBlock
    # SCAFFOLD:                     commands.append(block.input.get("command"))
    # SCAFFOLD:                     print("tool:  ", block.input.get("command"))
    # SCAFFOLD:                 elif isinstance(block, TextBlock):
    # SCAFFOLD:                     print("claude:", block.text)
    # SCAFFOLD:         elif isinstance(message, ...):   # ResultMessage
    # SCAFFOLD:             result = message
    # SCAFFOLD:             print("turns:", message.num_turns, "| stop:", message.stop_reason)
    # SCAFFOLD:
    # SCAFFOLD: run_async(main())
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("review.json still there:", Path("review.json").exists())

check("hook_blocks", commands=commands, result=result)

---

## Done

With the Agent SDK you write the task, the allowed tools and the hooks, not the loop; and anything that must never happen is a PreToolUse hook that returns deny - tested like any other code.